In [9]:
# Εγκατάσταση της βιβλιοθήκης
!pip install deepface

from deepface import DeepFace

# Η διαδρομή της φωτογραφίας μέσα από το συνδεδεμένο Drive
img_path = "/content/drive/MyDrive/Colab Notebooks/vhx0eub2kg461.jpg"

# Εκτέλεση του ArcFace
embedding_objs = DeepFace.represent(img_path = img_path, model_name = "ArcFace")

# Εξαγωγή και εκτύπωση των αποτελεσμάτων
embedding = embedding_objs[0]["embedding"]
print("\n" + "="*40)
print("Το embedding δημιουργήθηκε με επιτυχία!")
print("Μήκος διανύσματος (Διαστάσεις):", len(embedding))
print("Οι πρώτοι 5 αριθμοί είναι:", embedding[:5])
print("="*40)




Το embedding δημιουργήθηκε με επιτυχία!
Μήκος διανύσματος (Διαστάσεις): 512
Οι πρώτοι 5 αριθμοί είναι: [-0.038810454308986664, -0.016310401260852814, -0.020323272794485092, -0.3090369701385498, 0.044278182089328766]


In [10]:
import tensorflow as tf
from deepface import DeepFace

# Φέρνουμε το μοντέλο ArcFace στη μνήμη του Colab
model_obj = DeepFace.build_model("ArcFace")

# Παίρνουμε το καθαρό Keras μοντέλο από μέσα του
keras_model = model_obj.model

# Στήνουμε τον Converter χρησιμοποιώντας το Keras μοντέλο
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Εκτελούμε τη μετατροπή (εδώ γίνεται το Quantization)
tflite_model = converter.convert()

# Αποθηκεύουμε το συμπιεσμένο μοντέλο στο Colab
with open ("arcface_quantized.tflite", "wb") as f:
  f.write(tflite_model)

print("Το μοντέλο συμπιέστηκε και αποθηκεύτηκε ως 'arcface_quantized.tflite'!")

Το μοντέλο συμπιέστηκε και αποθηκεύτηκε ως 'arcface_quantized.tflite'!


In [11]:
import os

# Υπολογισμός μεγέθους σε bytes
original_size_mb = 137 # Το αρχικό μέγεθος που ξέρουμε από τη βιβλιογραφία
quantized_size_bytes = os.path.getsize("arcface_quantized.tflite")

# Μετατροπή σε megabytes
quantized_size_mb = quantized_size_bytes / (1024 * 1024)

print("="*40)
print(f"Αρχικό Μέγεθος Μοντέλου: {original_size_mb} MB")
print(f"Νέο Μέγεθος (Quantized): {quantized_size_mb:.2f} MB")
print(f"Ποσοστό Συμπίεσης: {((original_size_mb - quantized_size_mb) / original_size_mb) * 100:.1f}%")
print("="*40)

Αρχικό Μέγεθος Μοντέλου: 137 MB
Νέο Μέγεθος (Quantized): 32.75 MB
Ποσοστό Συμπίεσης: 76.1%


In [12]:
import time
import numpy as np
import tensorflow as tf

# Φόρτωση του κβαντισμένου μοντέλου στον TFLite Interpreter
interpreter = tf.lite.Interpreter(model_path="arcface_quantized.tflite")
interpreter.allocate_tensors()

# Πληροφορίες για τα input και output tensors του μοντέλου
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Δημιουργία μιας τυχαίας dummy εικόνας με τις σωστές διαστάσεις που ζητάει το μοντέλο
# (Αντί για την κανονική εικόνα, για να μετρήσουμε καθαρά τον χρόνο του interpreter)
input_shape = input_details[0]['shape']
dummy_input = np.array(np.random.random_sample(input_shape), dtype=np.float32)

# Warm-up run (Η πρώτη εκτέλεση είναι πάντα πιο αργή, την κάνουμε μια φορά εκτός μέτρησης)
interpreter.set_tensor(input_details[0]['index'], dummy_input)
interpreter.invoke()

# Μέτρηση ταχύτητας (τρέχουμε 10 επαναλήψεις για να βγάλουμε ακριβή μέσο όρο)
times = []
for _ in range(10):
  start_time = time.time()
  interpreter.set_tensor(input_details[0]['index'], dummy_input)
  interpreter.invoke()
  end_time = time.time()
  times.append(end_time - start_time)

# Υπολογισμός μέσου όρου σε χιλιοστά του δευτερολέπτου (ms)
avg_inference_time_ms = np.mean(times) * 1000

print("="*40)
print(f"Μέσος Χρόνος Εκτέλεσης (Inference Time): {avg_inference_time_ms:.2f} ms")
print("="*40)

Μέσος Χρόνος Εκτέλεσης (Inference Time): 280.67 ms
